In [2]:
import numpy as np

# Constants
NGASES = 3  # CH4, O2, CO2

# Henry's law constants (inverse formulation)
c_h_inv = np.array([1600.0, 1500.0, 2400.0])  # K - CH4, O2, CO2
kh_theta = np.array([714.29, 769.23, 29.4])  # L*atm/mol - CH4, O2, CO2
kh_tbase = 298.15  # K - base temperature

# Universal gas constant
rgasLatm = 0.08206  # L*atm/mol/K


def print_var(value, var_name, idate, j, s):
    """
    Debug printing function (mimics Fortran Call print_var).
    
    Parameters
    ----------
    value : float
        Value to print
    var_name : str
        Variable name
    idate : int or str
        Date identifier
    j : int
        Layer index
    s : int
        Gas species index (0-indexed in Python)
    """
    print(f"{var_name}: {value:.6e} (idate={idate}, j={j}, s={s})")


def henry_law_inverse(t_grnd, t_soisno, nl_soil, idate=None, 
                      debug=False, ngases=2):
    """
    Calculate dimensionless Henry's coefficients using inverse formulation.
    
    This uses the inverse formulation from Wania (4.12) and (4.21).
    Note: Only processes first 2 gases (s=1,2 in Fortran, s=0,1 in Python).
    
    Parameters
    ----------
    t_grnd : float
        Ground surface temperature [K]
    t_soisno : array_like
        Soil temperature array [K], should have at least nl_soil elements
    nl_soil : int
        Number of soil layers
    idate : int or str, optional
        Date identifier for debug printing
    debug : bool, optional
        If True, print debug information (default: False)
    ngases : int, optional
        Number of gases to process (default: 2, max: 3)
    
    Returns
    -------
    k_h_cc : ndarray
        Dimensionless Henry's coefficient [(mol/m3w) / (mol/m3g)]
        Shape: (nl_soil+1, ngases)
    k_h_inv_last : ndarray
        Last calculated k_h_inv values [L*atm/mol]
        Shape: (nl_soil+1, ngases)
    """
    # Initialize output arrays
    k_h_cc = np.zeros((nl_soil + 1, ngases))
    k_h_inv_last = np.zeros((nl_soil + 1, ngases))
    
    # Loop over soil layers (0 = surface, 1:nl_soil = soil layers)
    for j in range(nl_soil + 1):
        # Loop over first 2 gases only (matching Fortran s=1,2)
        for s in range(ngases):
            if j == 0:
                # Surface layer - use ground temperature
                # (4.12) Wania: k_h_inv = exp(-c_h_inv * (1/T - 1/T_base) + ln(kh_theta))
                k_h_inv = np.exp(-c_h_inv[s] * (1.0 / t_grnd - 1.0 / kh_tbase) + 
                                 np.log(kh_theta[s]))
                # (4.21) Wania: k_h_cc = T / k_h_inv * R
                k_h_cc[j, s] = t_grnd / k_h_inv * rgasLatm
                k_h_inv_last[j, s] = k_h_inv
            else:
                # Soil layer - use soil temperature
                k_h_inv = np.exp(-c_h_inv[s] * (1.0 / t_soisno[j - 1] - 1.0 / kh_tbase) + 
                                 np.log(kh_theta[s]))
                k_h_cc[j, s] = t_soisno[j - 1] / k_h_inv * rgasLatm
                k_h_inv_last[j, s] = k_h_inv
            
            # Debug printing (if enabled)
            # if debug and idate is not None:
            #     print_var(k_h_cc[j, s], 'ch4_tran k_h_cc(j,s)', idate, j, s)
            #     print_var(k_h_inv, 'ch4_tran k_h_inv', idate, j, s)
    
    return k_h_cc, k_h_inv_last


# Example usage
if __name__ == "__main__":
    # Example parameters
    nl_soil = 10  # 10 soil layers
    t_grnd = 285.0  # K
    t_soisno = np.linspace(285.0, 280.0, nl_soil)  # Temperature profile
    idate = 20250101  # Example date
    
    # Calculate Henry's coefficients (with debug output)
    k_h_cc, k_h_inv = henry_law_inverse(t_grnd, t_soisno, nl_soil, 
                                        idate=idate, debug=True)
    
    print("\n" + "="*60)
    print("Summary of Results:")
    print("="*60)
    print(f"k_h_cc shape: {k_h_cc.shape}")
    print(f"\nSurface (j=0) - Dimensionless Henry's coefficients:")
    print(f"  CH4: {k_h_cc[0, 0]:.6f}")
    print(f"  O2:  {k_h_cc[0, 1]:.6f}")
    print(f"\nFirst soil layer (j=1):")
    print(f"  CH4: {k_h_cc[1, 0]:.6f}")
    print(f"  O2:  {k_h_cc[1, 1]:.6f}")
    
    print(f"\nSurface (j=0) - Henry's constant (inverse) [L*atm/mol]:")
    print(f"  CH4: {k_h_inv[0, 0]:.6f}")
    print(f"  O2:  {k_h_inv[0, 1]:.6f}")


Summary of Results:
k_h_cc shape: (11, 2)

Surface (j=0) - Dimensionless Henry's coefficients:
  CH4: 0.041941
  O2:  0.038347

First soil layer (j=1):
  CH4: 0.041941
  O2:  0.038347

Surface (j=0) - Henry's constant (inverse) [L*atm/mol]:
  CH4: 557.621407
  O2:  609.876679
